<a href="https://colab.research.google.com/github/rmacasa-cloud/recurrent-pretraining/blob/main/notebooks/huginn_t4_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Oct  7 23:17:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/algoverse'
RUN  = f'{BASE}/data/2026-10-07'
os.environ['HF_HOME'] = f'{BASE}/hf_cache'
os.makedirs(RUN, exist_ok=True)
print(RUN)

Mounted at /content/drive
/content/drive/MyDrive/algoverse/data/2026-10-07


In [ ]:
!pip install -q lm-eval
!python -c "import lm_eval; print(lm_eval.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.5 MB/s eta 0:00:00
0.4.13


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "tomg-group-umd/huginn-0125"
tok = AutoTokenizer.from_pretrained(MODEL)

for dt in [torch.bfloat16, torch.float16]:
    try:
        t0 = time.perf_counter()
        m = AutoModelForCausalLM.from_pretrained(
            MODEL, trust_remote_code=True, torch_dtype=dt, device_map="cuda")
        ids = tok("What is 17 plus 25?", return_tensors="pt").to("cuda")
        out = m.generate(**ids, max_new_tokens=20, num_steps=16)
        print(f"{dt}: {time.perf_counter()-t0:.1f}s ->",
              repr(tok.decode(out[0], skip_special_tokens=True)[:200]))
        del m; torch.cuda.empty_cache()
    except Exception as e:
        print(f"{dt}: FAILED {type(e).__name__}: {str(e)[:300]}")
        torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

The repository tomg-group-umd/huginn-0125 contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/tomg-group-umd/huginn-0125 .
 You can inspect the repository content at https://hf.co/tomg-group-umd/huginn-0125.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


raven_config_minimal.py:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/tomg-group-umd/huginn-0125:
- raven_config_minimal.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/6.62k [00:00<?, ?B/s]

...

[transformers] A new version of the following files was downloaded from https://huggingface.co/tomg-group-umd/huginn-0125:
- raven_modeling_minimal.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json:   0%|          | 0.00/6.19k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

torch.bfloat16: FAILED AttributeError: 'list' object has no attribute 'keys'
torch.float16: FAILED AttributeError: 'list' object has no attribute 'keys'


In [ ]:
import torch, time

MODEL = "tomg-group-umd/huginn-0125"
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)

for name, dt in [("bfloat16", torch.bfloat16), ("float16", torch.float16)]:
    print("=" * 40, name)
    t0 = time.perf_counter()
    m = AutoModelForCausalLM.from_pretrained(
        MODEL, trust_remote_code=True, dtype=dt, device_map="cuda")
    load_s = time.perf_counter() - t0

    ids = tok("The capital of Westphalia is", return_tensors="pt").to("cuda")
    t1 = time.perf_counter()
    with torch.no_grad():
        out = m(**ids, num_steps=16)
    fwd_s = time.perf_counter() - t1

    logits = out.logits if hasattr(out, "logits") else out[0]
    top = logits[0, -1].argmax().item()
    print(f"load {load_s:.1f}s | forward {fwd_s:.2f}s")
    print(f"logits dtype={logits.dtype} nan={torch.isnan(logits).any().item()} "
          f"inf={torch.isinf(logits).any().item()}")
    print(f"next token -> {tok.decode([top])!r}")
    print(f"gpu mem {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

    del m, out, logits
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

======================================== bfloat16


AttributeError: 'list' object has no attribute 'keys'

In [ ]:
import shutil, os
shutil.rmtree('/content/drive/MyDrive/algoverse/hf_cache', ignore_errors=True)
os.environ['HF_HOME'] = '/content/hf_cache'
!df -h /content | tail -1

overlay         113G   48G   66G  42% /


In [ ]:
!pip install -q "transformers==4.49.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'

import torch, time, transformers
print("transformers", transformers.__version__)
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "tomg-group-umd/huginn-0125"
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

for name, dt in [("bfloat16", torch.bfloat16), ("float16", torch.float16)]:
    print("=" * 40, name)
    t0 = time.perf_counter()
    m = AutoModelForCausalLM.from_pretrained(
        MODEL, trust_remote_code=True, torch_dtype=dt, device_map="cuda")
    print(f"load {time.perf_counter()-t0:.1f}s")

    ids = tok("The capital of Westphalia is", return_tensors="pt").to("cuda")
    t1 = time.perf_counter()
    with torch.no_grad():
        out = m(**ids, num_steps=16)
    lg = out.logits if hasattr(out, "logits") else out[0]
    print(f"forward {time.perf_counter()-t1:.2f}s | nan={torch.isnan(lg).any().item()} "
          f"inf={torch.isinf(lg).any().item()} | next={tok.decode([lg[0,-1].argmax().item()])!r}")
    print(f"peak mem {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    del m, out, lg
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

transformers 4.49.0


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

load 459.3s
forward 3.35s | nan=False inf=False | next=' the'
peak mem 7.8GB
======================================== float16


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

load 72.8s
forward 0.68s | nan=False inf=False | next=' the'
peak mem 7.8GB


In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'
RUN = '/content/drive/MyDrive/algoverse/data/2026-10-07'
os.makedirs(RUN, exist_ok=True)

!lm_eval --model hf \
  --model_args pretrained=tomg-group-umd/huginn-0125,trust_remote_code=True,dtype=float16,mean_recurrence=4 \
  --tasks hellaswag --limit 20 --batch_size=1 \
  --output_path={RUN}/smoke

2026-10-07:23:46:29 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-07:23:46:45 INFO     [_cli.run:392] Selected Tasks: ['hellaswag']
2026-10-07:23:46:48 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-07:23:46:48 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'tomg-group-umd/huginn-0125', 'trust_remote_code': True, 'dtype': 'float16', 'mean_recurrence': 4}
2026-10-07:23:46:51 INFO     [models.huggingface:289] Using device 'cuda:0'
2026-10-07:23:46:52 INFO     [models.huggingface:717] Model type cannot be determined. Using default model type 'causal'
2026-10-07:23:46:52 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
I0000 00:00:1791416824.851102    9896 cudart_stub.cc:31] C

In [ ]:
import subprocess, time, json

def run(n, limit=5):
    cmd = ["lm_eval", "--model", "hf",
      "--model_args", f"pretrained=tomg-group-umd/huginn-0125,trust_remote_code=True,dtype=float16,mean_recurrence={n}",
      "--tasks", "gsm8k_cot", "--batch_size", "1",
      "--apply_chat_template", "True", "--fewshot_as_multiturn",
      "--system_instruction", "You are a helpful assistant that can assist users with mathematical reasoning.",
      "--limit", str(limit), "--output_path", f"{RUN}/gsm8k_N{n}_L{limit}"]
    t0 = time.perf_counter()
    r = subprocess.run(cmd, capture_output=True, text=True)
    el = time.perf_counter() - t0
    print(f"N={n} total={el:.1f}s per_ex={el/limit:.1f}s rc={r.returncode}", flush=True)
    if r.returncode: print(r.stderr[-1200:])
    return {"N": n, "limit": limit, "total_s": el, "per_example_s": el/limit}

results = []
for n in [4, 8, 16, 32]:
    results.append(run(n, limit=5))
    json.dump(results, open(f"{RUN}/timing.json", "w"), indent=2)

print()
for r in results:
    print(f'N={r["N"]:>2}: {r["per_example_s"]:.1f}s/ex -> {r["per_example_s"]*1319/3600:.1f}h full GSM8K (T4, fp16)')

N=4 total=146.0s per_ex=29.2s rc=0
N=8 total=217.4s per_ex=43.5s rc=0
N=16 total=308.9s per_ex=61.8s rc=0
N=32 total=120.4s per_ex=24.1s rc=1
ules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py", line 134, in forward
    return F.linear(input, self.weight, self.bias)
           ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 60.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 27.81 MiB is free. Process 6867 has 152.00 MiB memory in use. Including non-PyTorch memory, this process has 14.38 GiB memory in use. Of the allocated memory 12.18 GiB is allocated by PyTorch, and 2.08 GiB is reserved by PyTorch but unallocated. If reserv

In [4]:
import json
print(open(f"{RUN}/timing.json").read())
!ls -la {RUN}

[
  {
    "N": 4,
    "limit": 5,
    "total_s": 145.98836370299978,
    "per_example_s": 29.197672740599955
  },
  {
    "N": 8,
    "limit": 5,
    "total_s": 217.3702976489999,
    "per_example_s": 43.47405952979998
  },
  {
    "N": 16,
    "limit": 5,
    "total_s": 308.9151723680002,
    "per_example_s": 61.78303447360004
  },
  {
    "N": 32,
    "limit": 5,
    "total_s": 120.43352312700017,
    "per_example_s": 24.086704625400035
  }
]
total 25
drwx------ 6 root root 4096 Oct  8 00:03 .
drwx------ 3 root root 4096 Oct  7 23:18 ..
drwx------ 3 root root 4096 Oct  8 00:01 gsm8k_N16_L5
drwx------ 3 root root 4096 Oct  7 23:52 gsm8k_N4_L5
drwx------ 3 root root 4096 Oct  7 23:56 gsm8k_N8_L5
drwx------ 3 root root 4096 Oct  7 23:49 smoke
-rw------- 1 root root  448 Oct  8 00:03 timing.json


In [5]:
import json
p = f"{RUN}/timing.json"
d = json.load(open(p))
for r in d:
    if r["N"] == 32:
        r["per_example_s"] = None
        r["status"] = "OOM at 14.38/14.56 GiB, crashed 7s into example 1"
    else:
        r["load_s_est"] = 75
        r["per_example_s_adj"] = (r["total_s"] - 75) / r["limit"]
json.dump(d, open(p, "w"), indent=2)
print(open(p).read())

[
  {
    "N": 4,
    "limit": 5,
    "total_s": 145.98836370299978,
    "per_example_s": 29.197672740599955,
    "load_s_est": 75,
    "per_example_s_adj": 14.197672740599955
  },
  {
    "N": 8,
    "limit": 5,
    "total_s": 217.3702976489999,
    "per_example_s": 43.47405952979998,
    "load_s_est": 75,
    "per_example_s_adj": 28.47405952979998
  },
  {
    "N": 16,
    "limit": 5,
    "total_s": 308.9151723680002,
    "per_example_s": 61.78303447360004,
    "load_s_est": 75,
    "per_example_s_adj": 46.78303447360004
  },
  {
    "N": 32,
    "limit": 5,
    "total_s": 120.43352312700017,
    "per_example_s": null,
    "status": "OOM at 14.38/14.56 GiB, crashed 7s into example 1"
  }
]
